# EKS + KServe: Production GPU Inference on Kubernetes

Generate and validate Kubernetes manifests for LLM serving:
- **KServe InferenceService** — model deployment with autoscaling
- **Karpenter GPU NodePool** — just-in-time GPU provisioning
- **Volcano PodGroup** — gang scheduling for multi-node tensor parallelism
- **Health check client** — v2 inference protocol probes

In [ ]:
import sys
sys.path.insert(0, '../../..')

import json, yaml, time
from dataclasses import dataclass, field
from typing import Optional, List
import requests

## 1. KServe InferenceService YAML Generator

Generates v1beta1 InferenceService manifests with GPU resource requests, HPA annotations, and storage configuration.

In [ ]:
@dataclass
class InferenceServiceConfig:
    name: str
    model_uri: str
    runtime: str = "tritonserver"
    gpu_count: int = 1
    min_replicas: int = 1
    max_replicas: int = 4
    target_utilization: int = 70
    memory: str = "16Gi"
    cpu: str = "4"
    gpu_type: str = "nvidia.com/gpu"

def generate_inference_service(cfg: InferenceServiceConfig) -> dict:
    """Generate KServe InferenceService manifest."""
    return {
        "apiVersion": "serving.kserve.io/v1beta1",
        "kind": "InferenceService",
        "metadata": {"name": cfg.name, "annotations": {
            "serving.kserve.io/autoscalerClass": "hpa",
            "serving.kserve.io/targetUtilizationPercentage": str(cfg.target_utilization),
        }},
        "spec": {"predictor": {
            "minReplicas": cfg.min_replicas,
            "maxReplicas": cfg.max_replicas,
            "containers": [{
                "name": "kserve-container",
                "image": f"nvcr.io/nvidia/{cfg.runtime}:24.05-trtllm-python-py3",
                "resources": {
                    "limits": {cfg.gpu_type: str(cfg.gpu_count), "memory": cfg.memory, "cpu": cfg.cpu},
                    "requests": {cfg.gpu_type: str(cfg.gpu_count), "memory": cfg.memory, "cpu": cfg.cpu}
                },
                "env": [{"name": "MODEL_URI", "value": cfg.model_uri}],
            }],
        }}
    }

# Generate for different model sizes
configs = [
    InferenceServiceConfig("llama-7b", "s3://models/llama-7b", gpu_count=1, memory="24Gi"),
    InferenceServiceConfig("llama-70b", "s3://models/llama-70b", gpu_count=4, memory="160Gi", max_replicas=2),
    InferenceServiceConfig("mixtral-8x7b", "s3://models/mixtral", gpu_count=2, memory="96Gi", runtime="vllm"),
]
for cfg in configs:
    print(f"--- {cfg.name} ({cfg.gpu_count} GPU, {cfg.memory}) ---")
    print(yaml.dump(generate_inference_service(cfg), default_flow_style=False))

## 2. Karpenter GPU NodePool

Just-in-time GPU node provisioning — Karpenter watches pending pods and launches the right instance type automatically.

In [ ]:
def generate_karpenter_nodepool(name: str, instance_types: List[str], gpu_type: str = "nvidia.com/gpu",
                                 max_gpus: int = 32, zones: List[str] = None) -> dict:
    """Generate Karpenter v1beta1 NodePool for GPU inference workloads."""
    zones = zones or ["us-east-1a", "us-east-1b"]
    return {
        "apiVersion": "karpenter.sh/v1beta1",
        "kind": "NodePool",
        "metadata": {"name": name},
        "spec": {
            "template": {
                "metadata": {"labels": {"workload-type": "gpu-inference", "nodepool": name}},
                "spec": {
                    "requirements": [
                        {"key": "karpenter.k8s.aws/instance-type", "operator": "In", "values": instance_types},
                        {"key": "topology.kubernetes.io/zone", "operator": "In", "values": zones},
                        {"key": "karpenter.sh/capacity-type", "operator": "In", "values": ["on-demand"]},
                    ],
                    "nodeClassRef": {"name": f"{name}-class"},
                }
            },
            "limits": {gpu_type: str(max_gpus)},
            "disruption": {"consolidationPolicy": "WhenUnderutilized", "expireAfter": "168h"},
        }
    }

# Tiered GPU pools
pools = [
    ("gpu-small", ["g5.2xlarge", "g5.4xlarge"], 16),
    ("gpu-large", ["p4d.24xlarge", "p4de.24xlarge"], 64),
    ("gpu-inferentia", ["inf2.24xlarge", "inf2.48xlarge"], 48),
]
for name, instances, max_g in pools:
    print(f"--- {name}: {instances} (max {max_g} GPUs) ---")
    print(yaml.dump(generate_karpenter_nodepool(name, instances, max_gpus=max_g), default_flow_style=False))

## 3. Volcano Gang Scheduling PodGroup

For multi-node tensor/pipeline parallel models, all pods must be scheduled simultaneously — otherwise partial scheduling causes deadlocks.

In [ ]:
def generate_volcano_podgroup(name: str, num_workers: int, gpu_per_worker: int,
                               image: str, model_path: str, port: int = 8000) -> dict:
    """Generate Volcano PodGroup + Job for gang-scheduled multi-node inference."""
    total_gpus = num_workers * gpu_per_worker
    return {
        "apiVersion": "batch.volcano.sh/v1alpha1",
        "kind": "Job",
        "metadata": {"name": name},
        "spec": {
            "minAvailable": num_workers,
            "schedulerName": "volcano",
            "plugins": {"ssh": [], "svc": []},
            "queue": "default",
            "policies": [{"event": "PodEvicted", "action": "RestartJob"}],
            "tasks": [{
                "replicas": num_workers,
                "name": "worker",
                "template": {"metadata": {
                    "annotations": {"scheduling.volcano.sh/group-name": name}
                }, "spec": {
                    "containers": [{
                        "name": "inference",
                        "image": image,
                        "command": ["python", "-m", "vllm.entrypoints.openai.api_server",
                                    "--model", model_path,
                                    "--tensor-parallel-size", str(gpu_per_worker),
                                    "--pipeline-parallel-size", str(num_workers),
                                    "--port", str(port)],
                        "resources": {"limits": {"nvidia.com/gpu": str(gpu_per_worker)}},
                        "ports": [{"containerPort": port}],
                    }],
                    "restartPolicy": "OnFailure",
                }},
            }],
        }
    }

# Multi-node deployments
deployments = [
    ("llama-70b-tp", 2, 4, "vllm/vllm-openai:v0.5.0", "/models/llama-70b-hf"),
    ("mixtral-8x22b", 4, 8, "vllm/vllm-openai:v0.5.0", "/models/mixtral-8x22b"),
]
for name, workers, gpus, img, path in deployments:
    print(f"--- {name}: {workers} nodes x {gpus} GPUs = {workers * gpus} total ---")
    print(yaml.dump(generate_volcano_podgroup(name, workers, gpus, img, path), default_flow_style=False))

## 4. Health Check Client

Probes KServe endpoints using the v2 inference protocol — checks liveness, readiness, model status, and inference latency.

In [ ]:
class InferenceHealthChecker:
    """Health check client for KServe InferenceServices (v2 protocol)."""

    def __init__(self, base_url: str, model_name: str, timeout: int = 5):
        self.base_url = base_url.rstrip("/")
        self.model_name = model_name
        self.timeout = timeout

    def check_ready(self) -> dict:
        url = f"{self.base_url}/v2/models/{self.model_name}/ready"
        try:
            r = requests.get(url, timeout=self.timeout)
            return {"status": "ready" if r.status_code == 200 else "not_ready", "code": r.status_code}
        except requests.exceptions.RequestException as e:
            return {"status": "unreachable", "error": str(e)}

    def check_health(self) -> dict:
        """Full health: liveness + readiness + model status + latency probe."""
        return {
            "live": self._probe(f"{self.base_url}/v2/health/live"),
            "ready": self._probe(f"{self.base_url}/v2/health/ready"),
            "model": self.check_ready(),
            "latency_ms": self._latency_probe(),
        }

    def _probe(self, url: str) -> bool:
        try:
            return requests.get(url, timeout=self.timeout).status_code == 200
        except:
            return False

    def _latency_probe(self) -> Optional[float]:
        """Minimal inference request to measure response time."""
        url = f"{self.base_url}/v2/models/{self.model_name}/infer"
        payload = {"inputs": [{"name": "text_input", "shape": [1], "datatype": "BYTES", "data": ["ping"]}]}
        try:
            t0 = time.time()
            r = requests.post(url, json=payload, timeout=30)
            return round((time.time() - t0) * 1000, 1) if r.status_code == 200 else None
        except:
            return None

# Demo
checker = InferenceHealthChecker("http://llama-7b.default.svc.cluster.local", "llama-7b")
print(f"Model: {checker.model_name}")
print(f"Ready endpoint: {checker.base_url}/v2/models/{checker.model_name}/ready")
print(f"Infer endpoint: {checker.base_url}/v2/models/{checker.model_name}/infer")
# checker.check_health()  # Uncomment with real cluster

In [ ]:
def deploy_full_stack(name: str, model_uri: str, gpu_count: int, num_nodes: int = 1,
                       instance_types: List[str] = None, max_replicas: int = 4) -> dict:
    """Orchestrate: NodePool + InferenceService/Volcano + Health checker."""
    instance_types = instance_types or (["p4d.24xlarge"] if gpu_count >= 4 else ["g5.2xlarge", "g5.4xlarge"])
    manifests = {}

    manifests["nodepool"] = generate_karpenter_nodepool(f"gpu-{name}", instance_types, max_gpus=max_replicas * num_nodes * gpu_count)

    if num_nodes > 1:
        manifests["volcano_job"] = generate_volcano_podgroup(name, num_nodes, gpu_count, "vllm/vllm-openai:v0.5.0", model_uri)
    else:
        cfg = InferenceServiceConfig(name, model_uri, gpu_count=gpu_count, max_replicas=max_replicas, memory=f"{gpu_count * 24}Gi")
        manifests["inference_service"] = generate_inference_service(cfg)

    return manifests

# End-to-end example
stack = deploy_full_stack("llama-70b", "s3://models/llama-70b", gpu_count=8, num_nodes=2)
for resource, manifest in stack.items():
    print(f"=== {resource} ({manifest['kind']}) ===")
    print(yaml.dump(manifest, default_flow_style=False)[:400])
    print()

## Summary

| Component | Purpose | Key Config |
|-----------|---------|------------|
| KServe InferenceService | Model serving + autoscaling | `minReplicas`, `maxReplicas`, GPU limits |
| Karpenter NodePool | JIT GPU node provisioning | Instance types, capacity limits, disruption policy |
| Volcano PodGroup | Gang scheduling for multi-node | `minAvailable` = all-or-nothing scheduling |
| Health Checker | v2 protocol probes | Liveness, readiness, model-ready, latency |

**Production notes:**
- Set `consolidationPolicy: WhenUnderutilized` to reclaim idle GPU nodes
- Use `PodEvicted -> RestartJob` policy for Volcano to handle preemptions
- Health checks must verify model readiness, not just container liveness
- Gang scheduling prevents partial allocation deadlocks in pipeline-parallel setups